# Notebook 04 — Pengujian Hipotesis (Hypothesis Testing)

**Nama:** Nabila Nurfajriyasah  
**Role:** Member D — Hypothesis Testing  
**Research Question:** "Apakah rata-rata waktu penyelesaian issue di repository Kubernetes melebihi batas wajar 30 hari (1 bulan)?"
"Apakah rata-rata waktu merge Pull Request di repository Kubernetes berbeda secara signifikan dari 14 hari (2 minggu)?"
"Apakah rata-rata waktu penyelesaian issue secara signifikan lebih lama dibandingkan rata-rata waktu merge Pull Request di repository Kubernetes?"
**Dataset:** Kubernetes GitHub Issues & Pull Requests (dari Member A)

## AI Usage Disclosure

| # | Task | Tool | Prompt | How output was used |
|---|------|------|--------|---------------------|
| 1 | Constructing the structural function for the Z-test | Claude | "Generate a program code for Member D's task using the dataset prepared by Member A." | The output was utilized as the initial framework for hypothesis.py, where the z_test_one_sample and z_test_two_sample functions were directly implemented according to the specifications. |
| 2 | Validating the correctness of the Z-statistic formula | Claude | "Is this code correct and compliant with the specifications provided by Member D?" | Utilized to verify that both the mathematical formulas and the structural functions conform strictly to the defined specifications. |


In [ ]:
## Import & Persiapan Data

In [ ]:
import pandas as pd
import numpy as np
from hypothesis import (
    z_test_one_sample,
    z_test_two_sample,
    load_data,
    compute_resolution_days,
    compute_merge_days
)

In [ ]:
# Load data dari Member A
issues, prs = load_data()

# Hitung waktu resolusi
resolution_days = compute_resolution_days(issues)
merge_days      = compute_merge_days(prs)

# Statistik deskriptif
res_mean  = resolution_days.mean()
res_std   = resolution_days.std()
res_n     = len(resolution_days)

merge_mean = merge_days.mean()
merge_std  = merge_days.std()
merge_n    = len(merge_days)

print('Statistik Deskriptif')
print(f'Issues (closed) : n={res_n}, mean={res_mean:.2f} hari, std={res_std:.2f}')
print(f'PR (merged)     : n={merge_n}, mean={merge_mean:.2f} hari, std={merge_std:.2f}')

Statistik Deskriptif
Issues (closed) : n=708, mean=34.40 hari, std=50.53
PR (merged)     : n=2860, mean=16.29 hari, std=28.55


---

# UJI 1 — One-Sample Z-Test (Issues)

## Langkah 1: Rumuskan Hipotesis

**H₀:** Rata-rata waktu resolusi issue = 30 hari  
**Hₐ:** Rata-rata waktu resolusi issue > 30 hari
**Alasan memilih mu0 = 30 hari:** Karena 30 hari = 1 bulan, dianggap batas wajar untuk penyelesaian issue di project open-source  

## Langkah 2: Tentukan Tingkat Signifikansi

Tingkat signifikansi yang digunakan: **α = 0.05**  
  

## Langkah 3: Tentukan Uji Statistik

Uji yang digunakan: **One-Sample Z-Test**  
  

## Langkah 4: Hitung Statistik Z & P-Value

In [ ]:
result1 = z_test_one_sample(
    x_bar=res_mean,
    mu0=30,
    sigma=res_std,
    n=res_n,
    alternative='greater',
    alpha=0.05
)

for k, v in result1.items():
    print(f'{k:17}: {v}')

z_stat           : 2.319
p_value          : 0.0102
decision         : Reject H0
interpretation   : Dengan alpha=0.05, terdapat cukup bukti statistik untuk menolak H0 (z=2.3190, p=0.0102). Rata-rata sampel (34.4035) berbeda signifikan dari mu0=30.


## Langkah 5: Buat Keputusan

Karena p-value (0.0102) < alpha (0.05), maka keputusannya adalah **Reject H0**  

## Langkah 6: Interpretasi

Dengan tingkat signifikansi $\alpha = 0.05$, terdapat cukup bukti statistik untuk menolak $H_0$ ($z = 2.3190, p = 0.0102$). Rata-rata waktu resolusi sampel sebesar $34.40$ hari terbukti lebih besar secara signifikan dari $\mu_0 = 30$ hari. Hal ini menunjukkan bahwa rata-rata waktu penyelesaian issue melebihi $30$ hari (membutuhkan waktu lebih dari $1$ bulan).

---

# UJI 2 — One-Sample Z-Test (Pull Requests)

## Langkah 1: Rumuskan Hipotesis

**H₀:** Rata-rata waktu merge PR = 14 hari
**Hₐ:** Rata-rata waktu merge PR ≠ 14 hari  
**Alasan memilih mu0 = 14 hari:** Karena 14 hari = 2 minggu, dianggap batas wajar untuk review dan merge PR  

## Langkah 2: Tentukan Tingkat Signifikansi

Tingkat signifikansi yang digunakan: **α = 0.05**   

## Langkah 3: Tentukan Uji Statistik

Uji yang digunakan: **One-Sample Z-Test**  

## Langkah 4: Hitung Statistik Z & P-Value

In [ ]:
result2 = z_test_one_sample(
    x_bar=merge_mean,
    mu0=14,
    sigma=merge_std,
    n=merge_n,
    alternative='two-sided',
    alpha=0.05
)

for k, v in result2.items():
    print(f'{k:17}: {v}')

z_stat           : 4.2811
p_value          : 0.0
decision         : Reject H0
interpretation   : Dengan alpha=0.05, terdapat cukup bukti statistik untuk menolak H0 (z=4.2811, p=0.0000). Rata-rata sampel (16.2856) berbeda signifikan dari mu0=14.


## Langkah 5: Buat Keputusan

Karena p-value < alpha (0.05), maka keputusannya adalah **Reject H0**

## Langkah 6: Interpretasi

Dengan tingkat signifikansi $\alpha = 0.05$, terdapat cukup bukti statistik untuk menolak $H_0$ ($z = 4.281, p = 0.000$). Rata-rata sampel waktu merge PR 16.28 berbeda secara signifikan dari $\mu_0 = 14$ hari. Hal ini menunjukkan bahwa rata-rata waktu untuk me-merge PR tidak sama dengan $14$ hari (2 minggu).

---

# UJI 3 — Two-Sample Z-Test (Issues vs PR)

## Langkah 1: Rumuskan Hipotesis

**H₀:** Rata-rata waktu resolusi issue = rata-rata waktu merge PR/ 14 hari  
**Hₐ:** Rata-rata waktu resolusi issue > rata-rata waktu merge PR/ 14 hari    

## Langkah 2: Tentukan Tingkat Signifikansi

Tingkat signifikansi yang digunakan: **α = 0.05**   

## Langkah 3: Tentukan Uji Statistik

Uji yang digunakan: **Two-Sample Z-Test**

## Langkah 4: Hitung Statistik Z & P-Value

In [ ]:
result3 = z_test_two_sample(
    x_bar1=res_mean,
    x_bar2=merge_mean,
    sigma1=res_std,
    sigma2=merge_std,
    n1=res_n,
    n2=merge_n,
    alternative='greater',
    alpha=0.05
)

for k, v in result3.items():
    print(f'{k:17}: {v}')

z_stat           : 9.1853
p_value          : 0.0
decision         : Reject H0
interpretation   : Dengan alpha=0.05, terdapat cukup bukti statistik untuk menolak H0 (z=9.1853, p=0.0000). Terdapat perbedaan signifikan antara dua kelompok (mean1=34.4035 vs mean2=16.2856).


## Langkah 5: Buat Keputusan

Karena p-value < alpha (0.05), maka keputusannya adalah **Reject H0**

## Langkah 6: Interpretasi

Berdasarkan hasil uji z dua sampel, diperoleh nilai statistik uji sebesar z = 9,1853 dengan p-value = 0,0000. Karena nilai p-value lebih kecil dari tingkat signifikansi α = 0,05, maka hipotesis nol (H₀) ditolak. Dengan demikian, terdapat bukti statistik yang sangat kuat bahwa rata-rata kedua kelompok berbeda secara signifikan. Rata-rata kelompok pertama sebesar 34,4035 lebih tinggi dibandingkan rata-rata kelompok kedua sebesar 16,2856, sehingga dapat disimpulkan bahwa kelompok pertama memiliki nilai rata-rata yang secara signifikan lebih besar daripada kelompok kedua.

---



## Ringkasan Hasil Ketiga Uji

| Uji | H₀ | Hasil | Keputusan |
| :--- | :--- | :--- | :--- |
| **Uji 1** | Rata-rata waktu resolusi issue = 30 hari | z = 2.319, p = 0.0102 | **Reject H0** |
| **Uji 2** | Rata-rata waktu merge PR = 14 hari | z = 4.281, p = 0 | **Reject H0** |
| **Uji 3** | Waktu resolusi issue = waktu merge PR | z = 9.185, p = 0 | **Reject H0** |

---

